# Callbacks fastai

> Callbacks dédiés à la collecte du latent `zi`.


In [ ]:
#| default_exp callbacks


In [ ]:
#| export
"""fastai callbacks used by Tell Me Why."""


from typing import Any

import torch


class CollectLatentSpaceCallback:
    """fastai callback collecting `learn.model.zi` during validation/test."""

    order = 60

    def before_validate(self) -> None:
        self.learn.zi_valid = []

    def before_batch(self) -> None:
        if not self.training and not hasattr(self.learn, "zi_valid"):
            self.learn.zi_valid = []

    def after_batch(self) -> None:
        if self.training:
            return
        zi = getattr(self.learn.model, "zi", None)
        if zi is not None:
            self.learn.zi_valid.append(zi.detach().cpu())

    def after_validate(self) -> None:
        zi_valid = getattr(self.learn, "zi_valid", [])
        if isinstance(zi_valid, list):
            self.learn.zi_valid = torch.cat(zi_valid) if zi_valid else torch.empty(0)


def as_fastai_callback(callback: CollectLatentSpaceCallback) -> Any:
    """Wrap the lightweight callback as a real fastai `Callback`."""

    from fastai.callback.core import Callback

    if isinstance(callback, Callback):
        return callback

    class _FastaiCollectLatentSpaceCallback(Callback):
        order = callback.order

        def before_validate(self) -> None:
            callback.learn = self.learn
            callback.before_validate()

        def before_batch(self) -> None:
            callback.learn = self.learn
            callback.training = self.training
            callback.before_batch()

        def after_batch(self) -> None:
            callback.learn = self.learn
            callback.training = self.training
            callback.after_batch()

        def after_validate(self) -> None:
            callback.learn = self.learn
            callback.after_validate()

    return _FastaiCollectLatentSpaceCallback()
